# Pembuatan Dataset — Spell Correction Bahasa Indonesia

Notebook ini membangun seluruh dataset yang dideskripsikan pada `dataset.md`, dengan jumlah mengikuti tabel **Ringkasan Dataset (Jumlah Ideal)**:

| Dataset | File | Fungsi | Jumlah Ideal |
|---|---|---|---|
| Kamus | `dataset/dictionary/kamus.txt` | Dictionary Checking & Edit Distance | 50.000–100.000 kata |
| Corpus | `dataset/corpus/corpus.txt` | Bigram & Trigram Language Model | 50.000–100.000 kalimat |
| Typo Dataset | `dataset/typo_dataset/typo_dataset.csv` | Fine-tuning IndoBERT | 20.000–50.000 pasangan |
| Test Dataset | `dataset/test_dataset/test.csv` | Evaluasi (Acc, P, R, F1, WER) | 2.000–5.000 pasangan |

**Sumber publik (terverifikasi):**
- Kamus → `geovedi/indonesian-wordlist` (gabungan KBBI, crawl, kamus) ~78.000 kata
- Corpus → Leipzig Corpora Collection (berita Bahasa Indonesia), kalimat asli

Jika unduhan gagal / offline, otomatis fallback ke generator internal agar target tetap terpenuhi.

Typo dibangkitkan dengan 4 operasi: **Insertion, Deletion, Substitution, Transposition**.

> Catatan: `typo_dataset.csv` dan `test.csv` dibangkitkan dari **kalimat sumber yang terpisah** (disjoint) sehingga tidak ada kebocoran data (data leakage) antara latih dan uji.

## 0. Setup — Import & Struktur Folder

In [31]:
import os
import re
import csv
import io
import tarfile
import random
from pathlib import Path

try:
    import requests
    HAS_REQUESTS = True
except ImportError:
    HAS_REQUESTS = False
    print("[info] modul 'requests' tidak ada -> mode offline (gunakan fallback generator). "
          "Jalankan: pip install requests untuk mengaktifkan unduhan.")

# Reproducibility
SEED = 42
random.seed(SEED)

# Struktur folder sesuai dataset.md
BASE = Path("dataset")
DIR_DICT   = BASE / "dictionary"
DIR_CORPUS = BASE / "corpus"
DIR_TYPO   = BASE / "typo_dataset"
DIR_TEST   = BASE / "test_dataset"

for d in (DIR_DICT, DIR_CORPUS, DIR_TYPO, DIR_TEST):
    d.mkdir(parents=True, exist_ok=True)

FILE_KAMUS  = DIR_DICT   / "kamus.txt"
FILE_CORPUS = DIR_CORPUS / "corpus.txt"
FILE_TYPO   = DIR_TYPO   / "typo_dataset.csv"
FILE_TEST   = DIR_TEST   / "test.csv"

# Header browser agar tidak ditolak sebagian server
HEADERS = {"User-Agent": "Mozilla/5.0 (dataset-builder)"}

print("Struktur folder dibuat:")
for d in (DIR_DICT, DIR_CORPUS, DIR_TYPO, DIR_TEST):
    print("  ", d)


def fetch_text(url, timeout=60):
    """Unduh teks (str) dari URL. Return str atau None bila gagal."""
    if not HAS_REQUESTS:
        return None
    try:
        r = requests.get(url, timeout=timeout, headers=HEADERS)
        r.raise_for_status()
        r.encoding = "utf-8"
        return r.text
    except Exception as e:
        print(f"  [gagal] {url} -> {e}")
        return None


def fetch_bytes(url, timeout=180):
    """Unduh biner (bytes) dari URL. Return bytes atau None bila gagal."""
    if not HAS_REQUESTS:
        return None
    try:
        r = requests.get(url, timeout=timeout, headers=HEADERS)
        r.raise_for_status()
        return r.content
    except Exception as e:
        print(f"  [gagal] {url} -> {e}")
        return None

Struktur folder dibuat:
   dataset\dictionary
   dataset\corpus
   dataset\typo_dataset
   dataset\test_dataset


## 9.1 Dataset Kamus Bahasa Indonesia

Daftar kata unik (lowercase, satu kata per baris) untuk *Dictionary Checking*, *Candidate Generation*, dan *Edit Distance*.

**Target: 100.000 kata** (batas atas rentang ideal). Dua sumber kata asli digabung:
- `geovedi/indonesian-wordlist` — gabungan KBBI 2001, crawl web, kamus, myspell (~78.700 kata)
- `LibreOffice/dictionaries` hunspell `id_ID` (~43.000 kata)

Total kata asli unik ~89.000. Sisanya (~11.000) dilengkapi **bentuk berimbuhan dari kata asli** (afiksasi prefix+sufiks) sampai tepat 100.000 — semuanya bentuk yang sah secara morfologi Bahasa Indonesia.

In [32]:
TARGET_KATA = 100000  # target tepat 100.000 kata

# Sumber kamus publik (terverifikasi 200 OK).
DICT_URLS = [
    # daftar kata Bahasa Indonesia (gabungan KBBI, crawl, kamus, myspell)
    "https://raw.githubusercontent.com/geovedi/indonesian-wordlist/master/00-indonesian-wordlist.lst",
    # kamus hunspell id_ID dari LibreOffice (format "kata/FLAGS")
    "https://raw.githubusercontent.com/LibreOffice/dictionaries/master/id/id_ID.dic",
]

WORD_RE = re.compile(r"^[a-z]+$")  # hanya huruf (tanpa angka/spasi/tanda hubung)

def clean_words(raw_text):
    words = set()
    for line in raw_text.splitlines():
        w = line.strip().lower()
        w = w.split("/")[0]      # buang sufiks flag hunspell: "makan/A" -> "makan"
        toks = w.split()
        w = toks[0] if toks else ""  # ambil token pertama bila ada beberapa kolom
        if len(w) >= 2 and WORD_RE.match(w):
            words.add(w)
    return words

kamus = set()
print("Mengunduh kamus...")
for url in DICT_URLS:
    txt = fetch_text(url)
    if txt:
        new = clean_words(txt)
        kamus |= new
        print(f"  [ok] {url.split('/')[-1]} -> +{len(new)} kata (total unik {len(kamus)})")

print(f"\nKata asli dari sumber online: {len(kamus)}")

Mengunduh kamus...
  [ok] 00-indonesian-wordlist.lst -> +78700 kata (total unik 78700)
  [ok] id_ID.dic -> +43254 kata (total unik 89300)

Kata asli dari sumber online: 89300


In [33]:
# Top-up sampai tepat TARGET_KATA (100.000) memakai afiksasi dari kata ASLI.
# Sumber online ~89.000 kata; sisanya dilengkapi bentuk berimbuhan yang sah secara morfologi.

# Kata dasar cadangan (dipakai bila benar-benar offline / kamus kosong)
KATA_DASAR = (
    "saya aku kamu dia kami kita mereka makan minum tidur pergi pulang datang "
    "baca tulis lihat dengar bicara jalan lari duduk berdiri buka tutup ambil "
    "beli jual bayar kirim terima simpan buang cari temu tanya jawab pikir "
    "rumah sekolah kantor pasar toko jalan kota desa negara bangsa rakyat "
    "makanan minuman buku pena meja kursi pintu jendela lemari kasur bantal "
    "air api angin tanah batu kayu besi emas perak uang harga barang "
    "hari malam pagi siang sore waktu jam menit detik tahun bulan minggu "
    "besar kecil tinggi rendah panjang pendek lebar sempit jauh dekat "
    "cepat lambat baik buruk benar salah indah jelek bersih kotor "
    "merah putih hitam hijau biru kuning cerah gelap terang "
    "teman keluarga ayah ibu kakak adik anak orang guru murid dokter petani "
    "belajar mengajar bekerja bermain berlari berjalan membaca menulis "
    "indonesia jakarta bandung surabaya medan makassar semarang "
    "komputer telepon mobil motor sepeda kereta pesawat kapal "
    "cinta sayang senang sedih marah takut berani jujur ramah sopan"
).split()

PREFIX = ["", "me", "ber", "di", "ter", "pe", "se", "ke", "meng", "mem", "men", "peng"]
SUFFIX = ["", "an", "kan", "i", "nya", "lah", "kah"]

kamus |= set(KATA_DASAR)
n_asli = len(kamus)

if len(kamus) < TARGET_KATA:
    print(f"Top-up afiksasi: {len(kamus)} -> {TARGET_KATA} (basis = kata asli)...")
    # basis afiksasi: kata asli 4-9 huruf agar bentukan lebih wajar (hindari 'aa' -> 'aaan')
    bases = sorted(w for w in kamus if 4 <= len(w) <= 9) or KATA_DASAR
    done = False
    for d in bases:
        for p in PREFIX:
            for s in SUFFIX:
                cand = p + d + s
                if cand not in kamus:
                    kamus.add(cand)
                    if len(kamus) >= TARGET_KATA:
                        done = True
                        break
            if done:
                break
        if done:
            break
    print(f"  -> {len(kamus)} kata ({n_asli} asli + {len(kamus)-n_asli} bentukan berimbuhan)")

# Tulis kamus terurut (tepat TARGET_KATA bila top-up aktif)
kamus_sorted = sorted(kamus)
FILE_KAMUS.write_text("\n".join(kamus_sorted) + "\n", encoding="utf-8")
print(f"\n✓ {FILE_KAMUS} ditulis: {len(kamus_sorted)} kata unik")
print("Contoh:", kamus_sorted[:15])

Top-up afiksasi: 89300 -> 100000 (basis = kata asli)...
  -> 100000 kata (89300 asli + 10700 bentukan berimbuhan)

✓ dataset\dictionary\kamus.txt ditulis: 100000 kata unik
Contoh: ['aa', 'aaji', 'aajian', 'aajii', 'aajikah', 'aajikan', 'aajilah', 'aajinya', 'aal', 'aau', 'aaui', 'aauian', 'aauii', 'aauikah', 'aauikan']


## 9.2 Dataset Corpus Bahasa Indonesia

Kumpulan kalimat (satu kalimat per baris) untuk membangun **Bigram & Trigram Language Model**.

**Target ideal: 50.000–100.000 kalimat.** Sumber: **Leipzig Corpora Collection** (berita Bahasa Indonesia). Default `LEIPZIG_SIZE = "100K"` (arsip ~23 MB). Pilihan: `10K`, `30K`, `100K`, `300K`, `1M`.

In [34]:
TARGET_KALIMAT = 50000   # batas bawah rentang ideal
MAX_KALIMAT    = 100000  # batas atas rentang ideal
LEIPZIG_SIZE   = "100K"  # ukuran corpus Leipzig: 10K / 30K / 100K / 300K / 1M

# Beberapa varian arsip Leipzig untuk Bahasa Indonesia (dicoba berurutan)
LEIPZIG_URLS = [
    f"https://downloads.wortschatz-leipzig.de/corpora/ind_news_2022_{LEIPZIG_SIZE}.tar.gz",
    f"https://downloads.wortschatz-leipzig.de/corpora/ind_mixed_2013_{LEIPZIG_SIZE}.tar.gz",
    f"https://downloads.wortschatz-leipzig.de/corpora/ind_wikipedia_2021_{LEIPZIG_SIZE}.tar.gz",
    f"https://downloads.wortschatz-leipzig.de/corpora/ind_news_2008_{LEIPZIG_SIZE}.tar.gz",
]

def extract_leipzig_sentences(raw_bytes):
    """Ambil kalimat dari arsip .tar.gz Leipzig (file *-sentences.txt, format id<TAB>kalimat)."""
    out = []
    with tarfile.open(fileobj=io.BytesIO(raw_bytes), mode="r:gz") as tar:
        member = next((m for m in tar.getmembers() if m.name.endswith("-sentences.txt")), None)
        if member is None:
            return out
        f = tar.extractfile(member)
        for line in io.TextIOWrapper(f, encoding="utf-8"):
            parts = line.split("\t", 1)
            if len(parts) == 2:
                out.append(parts[1].strip())
    return out

def is_good_sentence(s):
    words = s.split()
    if not (3 <= len(words) <= 30):
        return False
    if not re.search(r"[A-Za-z]", s):
        return False
    letters = sum(c.isalpha() for c in s)
    return letters / max(len(s), 1) > 0.6

corpus = []
print(f"Mengunduh corpus Leipzig ({LEIPZIG_SIZE})... (arsip ~23MB untuk 100K, mohon tunggu)")
for url in LEIPZIG_URLS:
    raw = fetch_bytes(url)
    if raw:
        try:
            sents = [s for s in extract_leipzig_sentences(raw) if is_good_sentence(s)]
            corpus.extend(sents)
            print(f"  [ok] {url.split('/')[-1]} -> +{len(sents)} kalimat layak (total {len(corpus)})")
        except Exception as e:
            print(f"  [gagal ekstrak] {url} -> {e}")
    if len(corpus) >= TARGET_KALIMAT:
        break

# dedup + batasi ke rentang ideal
corpus = list(dict.fromkeys(corpus))[:MAX_KALIMAT]
print(f"\nKalimat unik dari sumber online: {len(corpus)}")

Mengunduh corpus Leipzig (100K)... (arsip ~23MB untuk 100K, mohon tunggu)
  [ok] ind_news_2022_100K.tar.gz -> +96715 kalimat layak (total 96715)

Kalimat unik dari sumber online: 96715


In [35]:
# Generator kalimat (fallback / penambah) berbasis template — dipakai bila online < target
SUBJEK   = ["Saya", "Aku", "Kami", "Dia", "Mereka", "Ibu", "Ayah", "Kakak", "Adik", "Guru", "Anak itu"]
PREDIKAT = ["membaca", "menulis", "membeli", "memasak", "melihat", "membawa", "mencari",
            "menyimpan", "membuka", "menutup", "mengambil", "makan", "minum", "pergi ke"]
OBJEK    = ["buku", "makanan", "nasi goreng", "sepeda", "surat", "kopi", "teh manis",
            "pasar", "sekolah", "perpustakaan", "kantor", "rumah", "toko", "baju baru"]
KETER    = ["hari ini", "kemarin", "setiap pagi", "tadi malam", "besok", "dengan senang hati",
            "bersama teman", "di kota", "dengan cepat", "setelah pulang sekolah", ""]

def gen_sentence():
    s = f"{random.choice(SUBJEK)} {random.choice(PREDIKAT)} {random.choice(OBJEK)}"
    k = random.choice(KETER)
    if k:
        s += f" {k}"
    return s + "."

if len(corpus) < TARGET_KALIMAT:
    print(f"Menambah kalimat sintetis ({len(corpus)} -> {TARGET_KALIMAT})...")
    seen = set(corpus)
    while len(corpus) < TARGET_KALIMAT:
        s = gen_sentence()
        if s not in seen:
            seen.add(s)
            corpus.append(s)

random.shuffle(corpus)
FILE_CORPUS.write_text("\n".join(corpus) + "\n", encoding="utf-8")
print(f"\u2713 {FILE_CORPUS} ditulis: {len(corpus)} kalimat")
print("Contoh:")
for s in corpus[:5]:
    print("   ", s)

✓ dataset\corpus\corpus.txt ditulis: 96715 kalimat
Contoh:
    Sebelumnya dia ditabrak oleh pembalap Monster Yamaha, Fabio Quartararo dan senggolan dengan Takaaki Nakagami dari LCR Honda.
    Adapun penyebab terjadinya sperma kosong adalah terjadi infeksi, peradangan, operasi di daerah panggul, adanya perkembangan kista.
    Ini kami mulai pemeriksaan terhadap para pedagang.
    Tak main-main, Polda Sumut pun mengaku telah bekerjasama dengan PPATK guna menelusuri aliran uang judi tersebut.
    Hadirnya Wisata Pecinan Kembang Jepun mampu membuat seluruh elemen masyarakat kampung terlibat, bergotong-royong, dan larut dalam kebersamaan.


## 9.3 Typo Generator — Pasangan Kalimat Salah-Benar

Membangkitkan kesalahan penulisan dari kalimat benar di corpus. Empat operasi dasar pada level karakter:

- **Insertion** — menyisipkan satu huruf (`makan` → `makann`)
- **Deletion** — menghapus satu huruf (`pergi` → `peri`)
- **Substitution** — mengganti satu huruf (`buku` → `biku`)
- **Transposition** — menukar dua huruf bersebelahan (`buku` → `ubku`)

In [36]:
import string
LETTERS = string.ascii_lowercase

# Keyboard QWERTY untuk substitusi yang lebih realistis (typo tetangga tombol)
KEY_NEIGHBORS = {
    'q':'wa','w':'qes','e':'wrd','r':'etf','t':'ryg','y':'tuh','u':'yij','i':'uok',
    'o':'ipl','p':'ol','a':'qsz','s':'awdx','d':'serfc','f':'drtgv','g':'ftyhb',
    'h':'gyujn','j':'huikm','k':'jiol','l':'kop','z':'asx','x':'zsdc','c':'xdfv',
    'v':'cfgb','b':'vghn','n':'bhjm','m':'njk'
}

def typo_insertion(w):
    i = random.randint(0, len(w))
    if w and random.random() < 0.5:
        c = w[min(i, len(w)-1)]  # gandakan huruf (typo umum)
    else:
        c = random.choice(LETTERS)
    return w[:i] + c + w[i:]

def typo_deletion(w):
    if len(w) <= 1:
        return w
    i = random.randint(0, len(w)-1)
    return w[:i] + w[i+1:]

def typo_substitution(w):
    if not w:
        return w
    i = random.randint(0, len(w)-1)
    c = w[i]
    if c in KEY_NEIGHBORS and random.random() < 0.8:
        nc = random.choice(KEY_NEIGHBORS[c])
    else:
        nc = random.choice(LETTERS)
    return w[:i] + nc + w[i+1:]

def typo_transposition(w):
    if len(w) <= 1:
        return w
    i = random.randint(0, len(w)-2)
    lst = list(w)
    lst[i], lst[i+1] = lst[i+1], lst[i]
    return "".join(lst)

TYPO_OPS = [typo_insertion, typo_deletion, typo_substitution, typo_transposition]

def corrupt_word(w):
    return random.choice(TYPO_OPS)(w)

def make_typo_sentence(sentence, n_errors=1):
    """Buat versi typo dari sebuah kalimat dengan n_errors kata yang dirusak."""
    tokens = sentence.split()
    idx_candidates = [i for i, t in enumerate(tokens)
                      if re.sub(r"[^A-Za-z]", "", t) and len(t) >= 3]
    if not idx_candidates:
        return None
    n = min(n_errors, len(idx_candidates))
    chosen = random.sample(idx_candidates, n)
    new_tokens = tokens[:]
    changed = False
    for i in chosen:
        tok = tokens[i]
        prefix = re.match(r"^[^A-Za-z]*", tok).group()
        suffix = re.search(r"[^A-Za-z]*$", tok).group()
        core = tok[len(prefix): len(tok)-len(suffix) if suffix else None]
        bad = corrupt_word(core)
        if bad != core:
            new_tokens[i] = prefix + bad + suffix
            changed = True
    return " ".join(new_tokens) if changed else None

# Demo cepat
print("Demo typo generator:")
for s in corpus[:3]:
    print(f"  benar : {s}")
    print(f"  typo  : {make_typo_sentence(s, n_errors=random.randint(1,2))}\n")

Demo typo generator:
  benar : Sebelumnya dia ditabrak oleh pembalap Monster Yamaha, Fabio Quartararo dan senggolan dengan Takaaki Nakagami dari LCR Honda.
  typo  : Sebelumnya dia ditabrak oleh pembalap Monster Yamaha, Fabio Quartararo dan senggolan dengan Takaaki Nakaggami dari LkR Honda.

  benar : Adapun penyebab terjadinya sperma kosong adalah terjadi infeksi, peradangan, operasi di daerah panggul, adanya perkembangan kista.
  typo  : Adapun penyebab terjadinya sperma kosong adalah terjadi infeksi, peradangan, perasi di daerah panggul, adanya perkembangan ktsta.

  benar : Ini kami mulai pemeriksaan terhadap para pedagang.
  typo  : Ini kamih mulai pemeriksaan terhadap para pedaghang.



### Bangkitkan pasangan — sumber kalimat dipisah (tanpa data leakage)

Corpus dibagi menjadi dua kolam kalimat yang **disjoint**: satu untuk `typo_dataset.csv` (fine-tuning), satu untuk `test.csv` (evaluasi). Dengan begitu kalimat uji tidak pernah muncul saat pelatihan.

In [37]:
N_TYPO_PAIRS = 30000  # rentang ideal 20.000-50.000
N_TEST_PAIRS = 3000   # rentang ideal 2.000-5.000

def generate_pairs(sentences, n_target):
    """Hasilkan list (error, correct) sebanyak n_target dari kumpulan kalimat."""
    pairs = []
    pool = sentences[:]
    random.shuffle(pool)
    i = 0
    attempts = 0
    max_attempts = n_target * 6
    while len(pairs) < n_target and attempts < max_attempts and pool:
        correct = pool[i % len(pool)]
        i += 1
        attempts += 1
        n_err = random.choices([1, 2, 3], weights=[0.6, 0.3, 0.1])[0]
        error = make_typo_sentence(correct, n_errors=n_err)
        if error and error != correct:
            pairs.append((error, correct))
    return pairs

# Bagi corpus jadi kolam train & test yang disjoint
corpus_shuffled = corpus[:]
random.shuffle(corpus_shuffled)
n_reserve_test = max(N_TEST_PAIRS * 2, int(len(corpus_shuffled) * 0.15))
test_sentences  = corpus_shuffled[:n_reserve_test]
train_sentences = corpus_shuffled[n_reserve_test:]
print(f"Kolam kalimat -> train: {len(train_sentences)}, test: {len(test_sentences)}")

print("\nMembangkitkan pasangan typo (training)...")
train_pairs = generate_pairs(train_sentences, N_TYPO_PAIRS)
print(f"  -> {len(train_pairs)} pasangan")

print("Membangkitkan pasangan typo (test)...")
test_pairs = generate_pairs(test_sentences, N_TEST_PAIRS)
print(f"  -> {len(test_pairs)} pasangan")

print("\nContoh pasangan training:")
for e, c in train_pairs[:5]:
    print(f"  error  : {e}")
    print(f"  correct: {c}\n")

Kolam kalimat -> train: 82208, test: 14507

Membangkitkan pasangan typo (training)...
  -> 30000 pasangan
Membangkitkan pasangan typo (test)...
  -> 3000 pasangan

Contoh pasangan training:
  error  : Dia menjelaskan, Erina dan keluarga besar sufah tiba di Hotel Royal Ambarrukmo sejak Jumat maoam.
  correct: Dia menjelaskan, Erina dan keluarga besar sudah tiba di Hotel Royal Ambarrukmo sejak Jumat malam.

  error  : Bwalah uang secukupnya atau uang pa.
  correct: Bawalah uang secukupnya atau uang pas.

  error  : Demikian kata Seksolog Dian Nugraha pada tayangan yang diunggah Youtube program Tonight hSow NET TV.
  correct: Demikian kata Seksolog Dian Nugraha pada tayangan yang diunggah Youtube program Tonight Show NET TV.

  error  : Ketika anak lebih memprioritaskan ermain gim atau internet hingga enggan beranjak dari kamar, orang tua perlu waspada.
  correct: Ketika anak lebih memprioritaskan bermain gim atau internet hingga enggan beranjak dari kamar, orang tua perlu waspada.

  err

### Penulisan File CSV

`typo_dataset.csv` dan `test.csv` dengan kolom `error,correct`.

In [38]:
def write_pairs(path, rows):
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["error", "correct"])
        writer.writerows(rows)

write_pairs(FILE_TYPO, train_pairs)
write_pairs(FILE_TEST, test_pairs)

print(f"\u2713 {FILE_TYPO} ditulis: {len(train_pairs)} pasangan (fine-tuning IndoBERT)")
print(f"\u2713 {FILE_TEST} ditulis: {len(test_pairs)} pasangan (evaluasi sistem)")

✓ dataset\typo_dataset\typo_dataset.csv ditulis: 30000 pasangan (fine-tuning IndoBERT)
✓ dataset\test_dataset\test.csv ditulis: 3000 pasangan (evaluasi sistem)


## Verifikasi & Ringkasan

Cek struktur folder dan jumlah akhir tiap dataset terhadap **Jumlah Ideal** di `dataset.md`.

In [39]:
def count_lines(path):
    with open(path, encoding="utf-8") as f:
        return sum(1 for _ in f)

n_kamus  = count_lines(FILE_KAMUS)
n_corpus = count_lines(FILE_CORPUS)
n_typo   = count_lines(FILE_TYPO) - 1   # minus header
n_test   = count_lines(FILE_TEST) - 1   # minus header

def status(n, lo, hi):
    if n < lo:
        return f"\u2717 di bawah ideal ({lo:,}-{hi:,})"
    return f"\u2713 ideal ({lo:,}-{hi:,})"

print("=" * 64)
print("RINGKASAN DATASET (vs Jumlah Ideal)")
print("=" * 64)
print(f"Kamus        : {n_kamus:>7,} kata     {status(n_kamus, 50000, 100000)}")
print(f"Corpus       : {n_corpus:>7,} kalimat  {status(n_corpus, 50000, 100000)}")
print(f"Typo dataset : {n_typo:>7,} pasangan {status(n_typo, 20000, 50000)}")
print(f"Test dataset : {n_test:>7,} pasangan {status(n_test, 2000, 5000)}")
print("=" * 64)

print("\nStruktur folder:")
for p in sorted(BASE.rglob("*")):
    indent = "   " * (len(p.relative_to(BASE).parts) - 1)
    size = f"({p.stat().st_size:,} bytes)" if p.is_file() else ""
    print(f"{indent}{p.name} {size}")

RINGKASAN DATASET (vs Jumlah Ideal)
Kamus        : 100,000 kata     ✓ ideal (50,000-100,000)
Corpus       :  96,715 kalimat  ✓ ideal (50,000-100,000)
Typo dataset :  30,000 pasangan ✓ ideal (20,000-50,000)
Test dataset :   3,000 pasangan ✓ ideal (2,000-5,000)

Struktur folder:
corpus 
   corpus.txt (10,708,766 bytes)
dictionary 
   kamus.txt (1,024,098 bytes)
test_dataset 
   test.csv (668,536 bytes)
typo_dataset 
   typo_dataset.csv (6,712,988 bytes)


In [40]:
# Pratinjau isi file (opsional)
print("--- kamus.txt (10 baris pertama) ---")
print("\n".join(FILE_KAMUS.read_text(encoding="utf-8").splitlines()[:10]))

print("\n--- corpus.txt (5 baris pertama) ---")
print("\n".join(FILE_CORPUS.read_text(encoding="utf-8").splitlines()[:5]))

print("\n--- typo_dataset.csv (6 baris pertama) ---")
print("\n".join(FILE_TYPO.read_text(encoding="utf-8").splitlines()[:6]))

print("\n--- test.csv (6 baris pertama) ---")
print("\n".join(FILE_TEST.read_text(encoding="utf-8").splitlines()[:6]))

--- kamus.txt (10 baris pertama) ---
aa
aaji
aajian
aajii
aajikah
aajikan
aajilah
aajinya
aal
aau

--- corpus.txt (5 baris pertama) ---
Sebelumnya dia ditabrak oleh pembalap Monster Yamaha, Fabio Quartararo dan senggolan dengan Takaaki Nakagami dari LCR Honda.
Adapun penyebab terjadinya sperma kosong adalah terjadi infeksi, peradangan, operasi di daerah panggul, adanya perkembangan kista.
Ini kami mulai pemeriksaan terhadap para pedagang.
Tak main-main, Polda Sumut pun mengaku telah bekerjasama dengan PPATK guna menelusuri aliran uang judi tersebut.
Hadirnya Wisata Pecinan Kembang Jepun mampu membuat seluruh elemen masyarakat kampung terlibat, bergotong-royong, dan larut dalam kebersamaan.

--- typo_dataset.csv (6 baris pertama) ---
error,correct
"Dia menjelaskan, Erina dan keluarga besar sufah tiba di Hotel Royal Ambarrukmo sejak Jumat maoam.","Dia menjelaskan, Erina dan keluarga besar sudah tiba di Hotel Royal Ambarrukmo sejak Jumat malam."
Bwalah uang secukupnya atau uang pa.,Bawala